# 🔀 Notebook 10 — Corrected Multimodal Fusion Model

This notebook is the **source of truth for fusion training**.

✅ It trains fusion using **real modality model outputs** from labeled validation/test videos.  
✅ It saves `fusion_model.pth` and `fusion_config.json`.  
✅ The final demo notebook should only **load** this saved fusion model and run inference.

In [6]:
# Cell 1 — Install & imports
!pip install -q torch torchvision librosa opencv-python-headless scikit-learn tqdm matplotlib

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json, shutil, urllib.request, warnings, random
from pathlib import Path

import cv2
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights

from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Device:", DEVICE)

Mounted at /content/drive
✅ Device: cpu


In [7]:
import os

DRIVE_ROOT = "/content/drive/MyDrive"

if not os.path.exists(DRIVE_ROOT):
    raise RuntimeError("Google Drive is not mounted. Please mount Drive again.")

print("✅ Drive mounted correctly")

✅ Drive mounted correctly


In [8]:
# Cell 2 — Paths & config
BASE_DIR = "/content/drive/MyDrive/deepfake-project"

MODEL_DIR = os.path.join(BASE_DIR, "models")
DATA_DIR  = os.path.join(BASE_DIR, "data")
FUSION_DATASET_PATH = os.path.join(BASE_DIR, "outputs", "fusion_training_scores.json")
FUSION_MODEL_PATH   = os.path.join(MODEL_DIR, "fusion_model.pth")
FUSION_CONFIG_PATH  = os.path.join(MODEL_DIR, "fusion_config.json")

TEST_VIDEOS_DIR = os.path.join(DATA_DIR, "test_videos")
TMP_DIR = "/tmp/fusion_training_work"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

IMG_SIZE = 224
MOUTH_SIZE = 96
SEQ_LEN = 10
SR = 16000
AUDIO_WIN = 0.2
N_FRAMES_SYNC = 5
EMB_DIM = 128

MODEL_PATHS = {
    "visual":   os.path.join(MODEL_DIR, "visual_cnn.pth"),
    "temporal": os.path.join(MODEL_DIR, "temporal_lstm.pth"),
    "audio":    os.path.join(MODEL_DIR, "audio_crnn.pth"),
    "lipsync":  os.path.join(MODEL_DIR, "lipsync_net.pth"),
    "freq":     os.path.join(MODEL_DIR, "freq_model.pth"),
}

print("✅ BASE_DIR:", BASE_DIR)
for k, v in MODEL_PATHS.items():
    print(f"{k:>9}:", "FOUND" if os.path.exists(v) else "MISSING", v)

✅ BASE_DIR: /content/drive/MyDrive/deepfake-project
   visual: FOUND /content/drive/MyDrive/deepfake-project/models/visual_cnn.pth
 temporal: FOUND /content/drive/MyDrive/deepfake-project/models/temporal_lstm.pth
    audio: FOUND /content/drive/MyDrive/deepfake-project/models/audio_crnn.pth
  lipsync: FOUND /content/drive/MyDrive/deepfake-project/models/lipsync_net.pth
     freq: FOUND /content/drive/MyDrive/deepfake-project/models/freq_model.pth


In [9]:
# Cell 3 — Preprocessing helpers
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

mouth_tf = transforms.Compose([
    transforms.Resize((MOUTH_SIZE, MOUTH_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

if not os.path.exists("/tmp/deploy.prototxt"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
        "/tmp/deploy.prototxt"
    )
if not os.path.exists("/tmp/face_model.caffemodel"):
    urllib.request.urlretrieve(
        "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
        "/tmp/face_model.caffemodel"
    )

face_net = cv2.dnn.readNetFromCaffe("/tmp/deploy.prototxt", "/tmp/face_model.caffemodel")

def reset_tmp():
    if os.path.exists(TMP_DIR):
        shutil.rmtree(TMP_DIR)
    os.makedirs(TMP_DIR, exist_ok=True)

def extract_faces_and_mouths(video_path, max_frames=40):
    reset_tmp()
    face_dir = os.path.join(TMP_DIR, "faces")
    mouth_dir = os.path.join(TMP_DIR, "mouths")
    os.makedirs(face_dir, exist_ok=True)
    os.makedirs(mouth_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25

    if total <= 0:
        cap.release()
        return [], [], []

    indices = np.linspace(0, total - 1, min(max_frames, total)).astype(int)
    face_paths, mouth_paths, time_points = [], [], []

    for out_idx, frame_idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if not ok:
            continue

        h, w = frame.shape[:2]
        blob = cv2.dnn.blobFromImage(cv2.resize(frame, (300,300)), 1.0, (300,300), (104,177,123))
        face_net.setInput(blob)
        dets = face_net.forward()

        best = None
        best_conf = 0.0
        for i in range(dets.shape[2]):
            conf = float(dets[0,0,i,2])
            if conf > best_conf:
                best_conf = conf
                best = dets[0,0,i,3:7]

        if best is None or best_conf < 0.45:
            continue

        x1, y1, x2, y2 = (best * np.array([w,h,w,h])).astype(int)
        x1, y1 = max(0,x1), max(0,y1)
        x2, y2 = min(w,x2), min(h,y2)

        face = frame[y1:y2, x1:x2]
        if face.size == 0:
            continue

        fh, fw = face.shape[:2]
        mouth = face[int(fh*0.55):fh, int(fw*0.18):int(fw*0.82)]

        face_path = os.path.join(face_dir, f"face_{out_idx:03d}.jpg")
        mouth_path = os.path.join(mouth_dir, f"mouth_{out_idx:03d}.jpg")
        cv2.imwrite(face_path, face)
        cv2.imwrite(mouth_path, mouth)

        face_paths.append(face_path)
        mouth_paths.append(mouth_path)
        time_points.append(float(frame_idx / fps))

    cap.release()
    return face_paths, mouth_paths, time_points

def extract_audio(video_path):
    wav_path = os.path.join(TMP_DIR, "audio.wav")
    cmd = f'ffmpeg -y -i "{video_path}" -vn -ac 1 -ar {SR} "{wav_path}" -loglevel error'
    status = os.system(cmd)
    return wav_path if status == 0 and os.path.exists(wav_path) else None

def fft_map_from_image(face_path, size=128):
    img = cv2.imread(face_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (size, size))
    f = np.fft.fft2(img)
    fshift = np.fft.fftshift(f)
    mag = np.log(np.abs(fshift) + 1.0)
    mag = (mag - mag.mean()) / (mag.std() + 1e-6)
    return mag.astype(np.float32)

def audio_mel(wav_path):
    if wav_path is None or not os.path.exists(wav_path):
        return None
    y, _ = librosa.load(wav_path, sr=SR)
    if len(y) < SR * 0.2:
        return None
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128)
    mel = librosa.power_to_db(mel, ref=np.max)
    mel = cv2.resize(mel, (128,128))
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    return mel.astype(np.float32)

In [10]:
# Cell 4 — Model definitions
class VisualCNN(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(1792, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )
    def forward(self, x):
        feats = self.features(x)
        pooled = self.pool(feats).flatten(1)
        return self.classifier(pooled).squeeze(1)

class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d(1)
    def forward(self, x):
        return self.pool(self.features(x)).flatten(1)

class TemporalNet(nn.Module):
    def __init__(self, input_dim=1792, hidden=256):
        super().__init__()
        self.lstm1 = nn.LSTM(input_dim, hidden, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(hidden*2, 128, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(256, 1)
        self.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        out, _ = self.lstm1(x)
        out, _ = self.lstm2(out)
        w = torch.softmax(self.attn(out), dim=1)
        ctx = (w * out).sum(dim=1)
        return self.fc(ctx).squeeze(1)

class AudioCRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.gru = nn.GRU(128*16, 128, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )
    def forward(self, x):
        f = self.cnn(x)
        B,C,H,W = f.shape
        f = f.permute(0,3,1,2).reshape(B, W, C*H)
        out, _ = self.gru(f)
        return self.fc(out[:,-1,:]).squeeze(1)

class AudioEncoder(nn.Module):
    def __init__(self, emb_dim=EMB_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(80,256,3,padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256,256,3,padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(256,emb_dim),
        )
    def forward(self, x):
        return F.normalize(self.net(x.squeeze(1)), dim=1)

class VideoEncoder(nn.Module):
    def __init__(self, n_frames=N_FRAMES_SYNC, emb_dim=EMB_DIM):
        super().__init__()
        self.conv3d = nn.Sequential(
            nn.Conv3d(3,32,(3,5,5),stride=(1,2,2),padding=(1,2,2)),
            nn.BatchNorm3d(32), nn.ReLU(),
        )
        self.conv2d = nn.Sequential(
            nn.Conv2d(32*n_frames,128,3,padding=1), nn.ReLU(),
            nn.Conv2d(128,256,3,padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256,emb_dim),
        )
    def forward(self, x):
        x = x.permute(0,2,1,3,4)
        f = self.conv3d(x)
        B,C,T,H,W = f.shape
        f = f.reshape(B, C*T, H, W)
        return F.normalize(self.conv2d(f), dim=1)

class SyncNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.audio_enc = AudioEncoder()
        self.video_enc = VideoEncoder()
    def forward(self, a, v):
        return F.cosine_similarity(self.audio_enc(a), self.video_enc(v))

class FreqCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.5), nn.Linear(256,64), nn.ReLU(), nn.Linear(64,1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

print("✅ Model classes ready.")

class FusionCalibrator(nn.Module):
    def __init__(self, n_modalities=5):
        super().__init__()
        # Input = 5 modality probabilities + 5 availability mask values
        self.net = nn.Sequential(
            nn.Linear(n_modalities * 2, 32),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

print("✅ Model classes ready.")

✅ Model classes ready.
✅ Model classes ready.


In [11]:
# Cell 5 — Load trained modality models
def load_model(cls, path, name):
    model = cls().to(DEVICE)
    if not os.path.exists(path):
        print(f"⚠️ Missing {name}: {path}")
        return None
    model.load_state_dict(torch.load(path, map_location=DEVICE), strict=True)
    model.eval()
    print(f"✅ Loaded {name}")
    return model

visual_model   = load_model(VisualCNN,   MODEL_PATHS["visual"],   "visual_cnn")
feature_model  = FeatureExtractor().to(DEVICE).eval()
temporal_model = load_model(TemporalNet, MODEL_PATHS["temporal"], "temporal_lstm")
audio_model    = load_model(AudioCRNN,   MODEL_PATHS["audio"],    "audio_crnn")
lipsync_model  = load_model(SyncNet,     MODEL_PATHS["lipsync"],  "lipsync_net")
freq_model     = load_model(FreqCNN,     MODEL_PATHS["freq"],     "freq_model")

✅ Loaded visual_cnn
✅ Loaded temporal_lstm
✅ Loaded audio_crnn
✅ Loaded lipsync_net
✅ Loaded freq_model


In [12]:
# Cell 5B — Missing modality preprocessing helpers
# These helpers must be defined before Cell 7 because predict_audio(), predict_lipsync(),
# and predict_frequency() call them during fusion dataset creation.

# Audio CRNN was trained on 128x128 mel-spectrogram maps from 3-second audio chunks.
AUDIO_SR = 16000
AUDIO_N_MELS = 128
AUDIO_N_FFT = 1024
AUDIO_HOP_LENGTH = 512
AUDIO_DURATION = 3.0
AUDIO_SPEC_SIZE = 128

# Frequency CNN input size
MAP_SIZE = 128

def compute_mel_spec(wav_path,
                     sr=AUDIO_SR,
                     n_mels=AUDIO_N_MELS,
                     n_fft=AUDIO_N_FFT,
                     hop=AUDIO_HOP_LENGTH,
                     duration=AUDIO_DURATION,
                     target_size=AUDIO_SPEC_SIZE):
    """Return one 128x128 normalized mel-spectrogram for AudioCRNN inference."""
    try:
        y, _ = librosa.load(wav_path, sr=sr, mono=True)
    except Exception as e:
        print(f"⚠️ Audio load failed: {wav_path} | {e}")
        return None

    samples = int(sr * duration)
    if len(y) < samples:
        y = np.pad(y, (0, samples - len(y)))
    else:
        y = y[:samples]

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop,
        n_mels=n_mels
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)

    # Match Notebook 04 preprocessing: resize to 128x128 and normalize to [0, 1].
    img = Image.fromarray((log_mel * 255).astype(np.uint8))
    img = img.resize((target_size, target_size))
    return (np.array(img, dtype=np.float32) / 255.0)

def get_mel_segment(wav_path, start_sec, duration=AUDIO_WIN, sr=SR, n_mels=80):
    """Return one short 80-channel mel segment for SyncNet lip-sync inference."""
    try:
        y, _ = librosa.load(wav_path, sr=sr, offset=start_sec, duration=duration, mono=True)
    except Exception as e:
        print(f"⚠️ Lip-sync audio segment load failed: {wav_path} | {e}")
        return np.zeros((n_mels, 7), dtype=np.float32)

    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, fmax=8000)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    return log_mel.astype(np.float32)

def compute_frequency_map(img_bgr, size=MAP_SIZE):
    """Return one normalized FFT magnitude map for Frequency CNN inference."""
    if img_bgr is None:
        return np.zeros((size, size), dtype=np.float32)

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    gray = cv2.resize(gray, (size, size))

    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    freq = 20 * np.log(np.abs(fshift) + 1)

    freq = (freq - freq.min()) / (freq.max() - freq.min() + 1e-8)
    return freq.astype(np.float32)

print("✅ compute_mel_spec(), get_mel_segment(), and compute_frequency_map() defined.")

✅ compute_mel_spec(), get_mel_segment(), and compute_frequency_map() defined.


In [13]:
# Cell 6 — Modality inference
def predict_visual(face_paths):
    if visual_model is None or len(face_paths) == 0:
        return 0.5, []
    probs = []
    with torch.no_grad():
        for fp in face_paths:
            img = Image.open(fp).convert("RGB")
            x = val_tf(img).unsqueeze(0).to(DEVICE)
            probs.append(float(torch.sigmoid(visual_model(x)).item()))
    return float(np.mean(probs)), probs

def predict_temporal(face_paths):
    if temporal_model is None or len(face_paths) < 4:
        return 0.5, []
    embs = []
    with torch.no_grad():
        for fp in face_paths:
            img = Image.open(fp).convert("RGB")
            x = val_tf(img).unsqueeze(0).to(DEVICE)
            embs.append(feature_model(x).squeeze(0).cpu().numpy().astype(np.float32))
    embs = np.stack(embs)

    if len(embs) < SEQ_LEN:
        pad = np.repeat(embs[-1][None, :], SEQ_LEN - len(embs), axis=0)
        embs = np.concatenate([embs, pad], axis=0)

    seq_probs = []
    starts = list(range(0, max(1, len(embs) - SEQ_LEN + 1), max(1, SEQ_LEN // 2))) or [0]
    with torch.no_grad():
        for s in starts:
            chunk = embs[s:s+SEQ_LEN]
            if len(chunk) < SEQ_LEN:
                pad = np.repeat(chunk[-1][None, :], SEQ_LEN - len(chunk), axis=0)
                chunk = np.concatenate([chunk, pad], axis=0)
            x = torch.tensor(chunk, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            seq_probs.append(float(torch.sigmoid(temporal_model(x)).item()))
    return float(np.mean(seq_probs)), seq_probs

def predict_audio(wav_path):
    if audio_model is None or wav_path is None or not os.path.exists(wav_path):
        return None
    mel = compute_mel_spec(wav_path)
    if mel is None:
        return None
    x = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return float(torch.sigmoid(audio_model(x)).item())

def load_mouth_clip(frame_paths, n_frames=N_FRAMES_SYNC):
    idxs = np.linspace(0, len(frame_paths)-1, n_frames, dtype=int)
    clip = []
    for i in idxs:
        img = Image.open(frame_paths[i]).convert("RGB")
        clip.append(mouth_tf(img))
    return torch.stack(clip)

def predict_lipsync(wav_path, mouth_paths):
    if lipsync_model is None or wav_path is None or len(mouth_paths) < N_FRAMES_SYNC:
        return None, []
    total_duration = librosa.get_duration(path=wav_path)
    n_windows = int(total_duration / AUDIO_WIN)
    if n_windows < 1:
        return None, []
    timeline = []
    with torch.no_grad():
        for ci in range(n_windows):
            start = ci * AUDIO_WIN
            mel = get_mel_segment(wav_path, start)
            mel_t = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)

            f_start = int(ci / max(n_windows, 1) * len(mouth_paths))
            f_end = min(len(mouth_paths), f_start + N_FRAMES_SYNC * 4)
            sub = mouth_paths[f_start:f_end]
            if len(sub) < N_FRAMES_SYNC:
                continue

            clip = load_mouth_clip(sub, N_FRAMES_SYNC).unsqueeze(0).to(DEVICE)
            sync_conf = float(torch.sigmoid(lipsync_model(mel_t, clip)).item())
            timeline.append((float(start), sync_conf))
    if not timeline:
        return None, []
    avg_sync = float(np.mean([s for _, s in timeline]))
    return float(1.0 - avg_sync), timeline

def predict_frequency(face_paths):
    if freq_model is None or len(face_paths) == 0:
        return 0.5, []
    probs = []
    with torch.no_grad():
        for fp in face_paths:
            img = cv2.imread(fp)
            fmap = compute_frequency_map(img)
            x = torch.tensor(fmap, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
            probs.append(float(torch.sigmoid(freq_model(x)).item()))
    return float(np.mean(probs)), probs

print("✅ Inference helpers ready.")
print('✅ Modality inference functions ready.')

✅ Inference helpers ready.
✅ Modality inference functions ready.


In [14]:
# Cell 7 — Build real fusion-training dataset from labeled videos
# Expected folders:
#   data/test_videos/real
#   data/test_videos/fake
#   data/test_videos/real-with-audio
#   data/test_videos/fake-with-audio

SPLITS = [
    ("real", 0),
    ("fake", 1),
    ("real-with-audio", 0),
    ("fake-with-audio", 1),
]

def score_one_video(video_path):
    face_paths, mouth_paths, _ = extract_faces_and_mouths(str(video_path), max_frames=40)
    wav_path = extract_audio(str(video_path))

    p_visual, _ = predict_visual(face_paths)
    p_temporal, _ = predict_temporal(face_paths)
    p_audio = predict_audio(wav_path)
    p_lipsync, _ = predict_lipsync(wav_path, mouth_paths)
    p_freq, _ = predict_frequency(face_paths)

    scores = {
        "Visual": float(p_visual),
        "Temporal": float(p_temporal),
        "Audio": float(p_audio) if p_audio is not None else None,
        "Lip-Sync": float(p_lipsync) if p_lipsync is not None else None,
        "Frequency": float(p_freq),
    }
    return scores

# Start fresh each run. This avoids accidentally training on an incomplete JSON from a failed/partial run.
if os.path.exists(FUSION_DATASET_PATH):
    os.remove(FUSION_DATASET_PATH)
    print(f"🧹 Removed old partial fusion dataset: {FUSION_DATASET_PATH}")

records = []

for folder, label in SPLITS:
    folder_path = os.path.join(TEST_VIDEOS_DIR, folder)
    if not os.path.isdir(folder_path):
        print(f"⚠️ Missing folder: {folder_path}")
        continue

    videos = sorted(list(Path(folder_path).glob("*.mp4")) + list(Path(folder_path).glob("*.mov")) + list(Path(folder_path).glob("*.avi")) + list(Path(folder_path).glob("*.mkv")))
    print(f"\n📁 {folder}: {len(videos)} videos")

    for vp in tqdm(videos):
        try:
            scores = score_one_video(vp)
            records.append({
                "video": str(vp),
                "label": int(label),
                "scores": scores
            })
        except Exception as e:
            print("⚠️ Skipped:", vp.name, "|", repr(e))

with open(FUSION_DATASET_PATH, "w") as f:
    json.dump(records, f, indent=2)

print(f"\n✅ Saved fusion training scores: {FUSION_DATASET_PATH}")
print("Total videos:", len(records))
print("Labels:", {0: sum(r['label']==0 for r in records), 1: sum(r['label']==1 for r in records)})

🧹 Removed old partial fusion dataset: /content/drive/MyDrive/deepfake-project/outputs/fusion_training_scores.json

📁 real: 40 videos


100%|██████████| 40/40 [12:33<00:00, 18.85s/it]



📁 fake: 40 videos


100%|██████████| 40/40 [11:56<00:00, 17.91s/it]



📁 real-with-audio: 25 videos


100%|██████████| 25/25 [14:59<00:00, 35.97s/it]



📁 fake-with-audio: 25 videos


100%|██████████| 25/25 [26:25<00:00, 63.42s/it]


✅ Saved fusion training scores: /content/drive/MyDrive/deepfake-project/outputs/fusion_training_scores.json
Total videos: 130
Labels: {0: 65, 1: 65}


In [15]:
# Cell 8 — Convert scores to fusion features
MODALITY_ORDER = ["Visual", "Temporal", "Audio", "Lip-Sync", "Frequency"]

def make_fusion_feature(scores):
    probs, mask = [], []
    for m in MODALITY_ORDER:
        v = scores.get(m)
        if v is None:
            probs.append(0.5)   # neutral value when missing
            mask.append(0.0)    # tells model this modality was unavailable
        else:
            probs.append(float(v))
            mask.append(1.0)
    return np.array(probs + mask, dtype=np.float32)

X = np.stack([make_fusion_feature(r["scores"]) for r in records])
y = np.array([r["label"] for r in records], dtype=np.float32)

assert len(set(y.tolist())) == 2, "Need both REAL and FAKE videos to train fusion properly."

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

print("✅ Feature shape:", X.shape)
print("Train:", len(X_train), "Val:", len(X_val))

✅ Feature shape: (130, 10)
Train: 97 Val: 33


In [16]:
# Cell 9 — Train corrected fusion model on real modality outputs
fusion_model = FusionCalibrator(n_modalities=len(MODALITY_ORDER)).to(DEVICE)

Xtr = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
ytr = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
Xva = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
yva = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=1e-3, weight_decay=1e-4)

best_auc = -1
best_state = None

for epoch in range(1, 301):
    fusion_model.train()
    optimizer.zero_grad()
    logits = fusion_model(Xtr)
    loss = criterion(logits, ytr)
    loss.backward()
    optimizer.step()

    fusion_model.eval()
    with torch.no_grad():
        val_probs = torch.sigmoid(fusion_model(Xva)).cpu().numpy()

    val_auc = roc_auc_score(y_val, val_probs)
    val_acc = accuracy_score(y_val, (val_probs >= 0.5).astype(int))

    if val_auc > best_auc:
        best_auc = val_auc
        best_state = {k: v.detach().cpu().clone() for k, v in fusion_model.state_dict().items()}

    if epoch % 50 == 0:
        print(f"Epoch {epoch:03d} | loss={loss.item():.4f} | val_acc={val_acc:.4f} | val_auc={val_auc:.4f}")

fusion_model.load_state_dict(best_state)
torch.save(fusion_model.state_dict(), FUSION_MODEL_PATH)

config = {
    "model_type": "FusionCalibrator",
    "modalities": MODALITY_ORDER,
    "input_format": "probabilities_plus_availability_mask",
    "threshold": 0.5,
    "missing_probability_value": 0.5,
    "trained_on": FUSION_DATASET_PATH,
}
with open(FUSION_CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

print("\n✅ Saved corrected fusion model:", FUSION_MODEL_PATH)
print("✅ Saved fusion config:", FUSION_CONFIG_PATH)
print("Best validation AUC:", round(float(best_auc), 4))

Epoch 050 | loss=0.6376 | val_acc=0.8788 | val_auc=0.9853
Epoch 100 | loss=0.5040 | val_acc=0.8788 | val_auc=0.9669
Epoch 150 | loss=0.4023 | val_acc=0.8788 | val_auc=0.9816
Epoch 200 | loss=0.3492 | val_acc=0.9394 | val_auc=0.9890
Epoch 250 | loss=0.3364 | val_acc=0.9394 | val_auc=0.9853
Epoch 300 | loss=0.3016 | val_acc=0.9394 | val_auc=0.9853

✅ Saved corrected fusion model: /content/drive/MyDrive/deepfake-project/models/fusion_model.pth
✅ Saved fusion config: /content/drive/MyDrive/deepfake-project/models/fusion_config.json
Best validation AUC: 0.989


In [17]:
# Cell 10 — Final validation report
fusion_model.eval()
with torch.no_grad():
    probs = torch.sigmoid(fusion_model(Xva)).cpu().numpy()

preds = (probs >= 0.5).astype(int)

print("Accuracy:", round(accuracy_score(y_val, preds), 4))
print("AUC:", round(roc_auc_score(y_val, probs), 4))
print("\nConfusion matrix [REAL, FAKE]:")
print(confusion_matrix(y_val, preds))
print("\nClassification report:")
print(classification_report(y_val, preds, target_names=["REAL", "FAKE"]))

print("\nSample predictions:")
for i in range(min(10, len(X_val))):
    print(f"label={int(y_val[i])} prob_fake={probs[i]:.4f} pred={int(preds[i])}")

Accuracy: 0.8788
AUC: 0.989

Confusion matrix [REAL, FAKE]:
[[17  0]
 [ 4 12]]

Classification report:
              precision    recall  f1-score   support

        REAL       0.81      1.00      0.89        17
        FAKE       1.00      0.75      0.86        16

    accuracy                           0.88        33
   macro avg       0.90      0.88      0.88        33
weighted avg       0.90      0.88      0.88        33


Sample predictions:
label=1 prob_fake=0.5292 pred=1
label=0 prob_fake=0.4632 pred=0
label=1 prob_fake=0.5315 pred=1
label=0 prob_fake=0.4658 pred=0
label=0 prob_fake=0.4375 pred=0
label=0 prob_fake=0.4704 pred=0
label=1 prob_fake=0.5321 pred=1
label=0 prob_fake=0.4587 pred=0
label=0 prob_fake=0.4607 pred=0
label=1 prob_fake=0.5091 pred=1
